# Analisar resultados (CPU)

Lê as predições de validação salvas por todos os treinos e compara as
execuções, sem treinar nem prever de novo. Ambiente só com CPU basta.

Referência: E01 (B0 448, campanha de outubro), validação por paciente com
sensibilidade 0,90: ROC-AUC 0,908 e especificidade 0,671; com TTA de 4
variantes, 0,913 e 0,711. A especificidade por paciente é a métrica
principal; tudo aqui é validação (o teste fica para o modelo final).

In [ ]:
import os
import subprocess
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"
REVISAO_CODIGO = "origin/main"

if not os.path.exists(PASTA_REPOSITORIO):
    subprocess.run(["git", "clone", URL_REPOSITORIO, PASTA_REPOSITORIO], check=True)

subprocess.run(["git", "-C", PASTA_REPOSITORIO, "fetch", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", PASTA_REPOSITORIO, "checkout", "--detach", REVISAO_CODIGO], check=True)

sys.path.insert(0, os.path.join(PASTA_REPOSITORIO, "src"))

# Testes das análises (dados sintéticos): conferem o código antes de usá-lo
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=PASTA_REPOSITORIO,
    env=dict(os.environ, PYTHONPATH=os.path.join(PASTA_REPOSITORIO, "src")),
    check=True
)

In [ ]:
from retinopatia import ambiente_colab, analise

ambiente_colab.montar_drive()

execucoes = analise.listar_execucoes()
print(len(execucoes), "execuções com predições de validação da Fase 2")

## Todas as execuções

Uma linha por execução e modo de TTA (`sem_tta`, `tta2`, `tta4`), ordenada pela
especificidade por paciente. `kappa_grau` é o kappa quadrático da cabeça de grau
(Kaggle 2015, 1º lugar: 0,85 no teste da competição).

In [ ]:
comparativa = analise.tabela_comparativa(execucoes)
comparativa.round(4)

## Dois olhos e cabeças, numa execução

Escolha a execução e o modo. As combinações são ajustadas por validação cruzada
entre pacientes da validação e medidas nos pacientes que ficaram de fora.

In [ ]:
EXECUCAO = comparativa.loc[0, "execucao"]
MODO = "sem_tta"

tabela = execucoes[EXECUCAO][MODO]

print(EXECUCAO, MODO)
display(analise.combinador_paciente(tabela).round(4))
display(analise.misturar_cabecas(tabela).round(4))

## Teto de um ensemble

Média das execuções escolhidas (por exemplo, as melhores de cada rodada). Não é
o modelo do celular; mostra quanto um professor ou ensemble poderia acrescentar.

In [ ]:
ESCOLHIDAS = list(comparativa["execucao"].unique()[:3])

media = analise.ensemble(execucoes[nome][MODO] for nome in ESCOLHIDAS)

print(ESCOLHIDAS)
analise.metricas(media)